# Quality Window

In [ ]:
import sys
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

sys.path.insert(0, os.path.abspath(".."))

warnings.filterwarnings("ignore")

SLURRIES = ["G50", "G45", "G40", "G40+IPA"]

In [ ]:
from heavyedge import ProfileData

X = pd.read_csv("../../_temp/v1/X.csv", index_col=[0, 1, 2])
Xpred = pd.read_csv("../../_temp/v1/Xpred_2D.csv", index_col=[0, 1, 2])
Delaunay = pd.read_csv("../../_temp/v1/delaunay.Xpred_2D.csv", index_col=[0, 1, 2])

X_unique = pd.read_csv("../../_temp/v1/Xunique.csv").set_index("slurry")

with ProfileData("../../_temp/v1/profiles.h5") as data:
    Ys, _, _ = data[:]
    Ys /= np.sum(Ys, axis=1, keepdims=True)

assert len(X_unique) == len(Ys), "Xunique and profile rows must be aligned."

In [ ]:
columns = ["slurry", "cosine_of_contact_angle"]
cos_thetas = X["cosine_of_contact_angle"].reset_index()[columns]
slurry_map = cos_thetas.drop_duplicates().set_index("cosine_of_contact_angle")["slurry"]

slurries = Xpred["cosine_of_contact_angle"].map(slurry_map)
unique_slurries = [s for s in SLURRIES if s in slurries.unique()]

## GPR (deterministic)

In [ ]:
pred_gpr = pd.read_csv("../../benchmarks/v1/gpr.Xpred_2D.csv").pivot(
    index=["index", "batch"],
    columns="target",
)

### H

In [ ]:
H_THRESHOLD = 1.1

In [ ]:
target = "H"
mean = pred_gpr["latent_mean"][target]
quality_window = mean < H_THRESHOLD
levels = [-0.5, 0.5, 1.5]
cmap = mcolors.ListedColormap(["lightgray", "tab:blue"])
norm = mcolors.BoundaryNorm(levels, cmap.N)

fig, axes = plt.subplots(1, len(unique_slurries), sharex=True, sharey=True)

for slurry, ax in zip(unique_slurries, axes):
    ok_pred = slurries == slurry
    this_Xpred_df = Xpred[ok_pred]
    this_Xpred = this_Xpred_df.to_xarray().to_array().values
    this_quality_window = quality_window[ok_pred.values].to_numpy(dtype=int)

    delaunay = Delaunay[ok_pred.values].values.reshape(this_Xpred.shape[1:])
    delaunay_masked = np.full_like(delaunay, np.nan, dtype=float)

    contour = ax.contourf(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        this_quality_window.reshape(this_Xpred.shape[1:]).squeeze(axis=-1),
        cmap=cmap,
        norm=norm,
        levels=levels,
    )

    this_X = X_unique[X_unique.index.get_level_values("slurry") == slurry]
    ax.scatter(
        this_X["gap_to_thickness_ratio"],
        this_X["capillary_number"],
        color="black",
        s=15,
        zorder=3,
    )

    ax.contour(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        delaunay.squeeze(axis=-1).astype(float),
        levels=[0.5],
        colors="k",
    )

    (cos_theta,) = this_Xpred_df["cosine_of_contact_angle"].unique()
    ax.set_title(f"Cos θ={cos_theta:.2f}")

fig.tight_layout(rect=[0.02, 0.05, 1.0, 0.8])
cbar_ax = fig.add_axes([0.18, 0.82, 0.64, 0.04])
cbar = fig.colorbar(contour, cax=cbar_ax, orientation="horizontal", ticks=[0, 1])
cbar.set_ticklabels([f"H ≥ {H_THRESHOLD:g}", f"H < {H_THRESHOLD:g}"])
cbar.set_label("H quality window", labelpad=6)
cbar.ax.xaxis.set_ticks_position("top")
cbar.ax.xaxis.set_label_position("top")
cbar.ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)

fig.supxlabel("Rgt")
fig.supylabel("Ca")
fig.show()

### phi

In [ ]:
phi_THRESHOLD = 0.25

In [ ]:
target = "phi_1"
mean = pred_gpr["latent_mean"][target]
quality_window = mean < phi_THRESHOLD
levels = [-0.5, 0.5, 1.5]
cmap = mcolors.ListedColormap(["lightgray", "tab:blue"])
norm = mcolors.BoundaryNorm(levels, cmap.N)

fig, axes = plt.subplots(1, len(unique_slurries), sharex=True, sharey=True)

for slurry, ax in zip(unique_slurries, axes):
    ok_pred = slurries == slurry
    this_Xpred_df = Xpred[ok_pred]
    this_Xpred = this_Xpred_df.to_xarray().to_array().values
    this_quality_window = quality_window[ok_pred.values].to_numpy(dtype=int)

    delaunay = Delaunay[ok_pred.values].values.reshape(this_Xpred.shape[1:])
    delaunay_masked = np.full_like(delaunay, np.nan, dtype=float)

    contour = ax.contourf(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        this_quality_window.reshape(this_Xpred.shape[1:]).squeeze(axis=-1),
        cmap=cmap,
        norm=norm,
        levels=levels,
    )

    ax.contour(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        delaunay.squeeze(axis=-1).astype(float),
        levels=[0.5],
        colors="k",
    )

    this_X = X_unique[X_unique.index.get_level_values("slurry") == slurry]
    ax.scatter(
        this_X["gap_to_thickness_ratio"],
        this_X["capillary_number"],
        color="black",
        s=15,
        zorder=3,
    )

    (cos_theta,) = this_Xpred_df["cosine_of_contact_angle"].unique()
    ax.set_title(f"Cos θ={cos_theta:.2f}")

fig.tight_layout(rect=[0.02, 0.05, 1.0, 0.8])
cbar_ax = fig.add_axes([0.18, 0.82, 0.64, 0.04])
cbar = fig.colorbar(contour, cax=cbar_ax, orientation="horizontal", ticks=[0, 1])
cbar.set_ticklabels([f"φ ≥ {phi_THRESHOLD:g}", f"φ < {phi_THRESHOLD:g}"])
cbar.set_label("φ quality window", labelpad=6)
cbar.ax.xaxis.set_ticks_position("top")
cbar.ax.xaxis.set_label_position("top")
cbar.ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)

fig.supxlabel("Rgt")
fig.supylabel("Ca")
fig.show()

### Combined

In [ ]:
H_mean = pred_gpr["latent_mean"]["H"]
phi_mean = pred_gpr["latent_mean"]["phi_1"]
combined_window = (H_mean < H_THRESHOLD) & (phi_mean < phi_THRESHOLD)

levels = [-0.5, 0.5, 1.5]
cmap = mcolors.ListedColormap(["lightgray", "tab:blue"])
norm = mcolors.BoundaryNorm(levels, cmap.N)

fig, axes = plt.subplots(1, len(unique_slurries), sharex=True, sharey=True)

for slurry, ax in zip(unique_slurries, axes):
    ok_pred = slurries == slurry
    this_Xpred_df = Xpred[ok_pred]
    this_Xpred = this_Xpred_df.to_xarray().to_array().values
    this_combined_window = combined_window[ok_pred.values].to_numpy(dtype=int)

    delaunay = Delaunay[ok_pred.values].values.reshape(this_Xpred.shape[1:])

    contour = ax.contourf(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        this_combined_window.reshape(this_Xpred.shape[1:]).squeeze(axis=-1),
        cmap=cmap,
        norm=norm,
        levels=levels,
    )

    ax.contour(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        delaunay.squeeze(axis=-1).astype(float),
        levels=[0.5],
        colors="k",
    )

    this_X = X_unique[X_unique.index.get_level_values("slurry") == slurry]
    ax.scatter(
        this_X["gap_to_thickness_ratio"],
        this_X["capillary_number"],
        color="black",
        s=15,
        zorder=3,
    )

    (cos_theta,) = this_Xpred_df["cosine_of_contact_angle"].unique()
    ax.set_title(f"Cos θ={cos_theta:.2f}")

fig.tight_layout(rect=[0.02, 0.05, 1.0, 0.8])
cbar_ax = fig.add_axes([0.18, 0.82, 0.64, 0.04])
cbar = fig.colorbar(contour, cax=cbar_ax, orientation="horizontal", ticks=[0, 1])
cbar.set_ticklabels(
    [
        "Outside combined window",
        f"H < {H_THRESHOLD:g} and φ < {phi_THRESHOLD:g}",
    ]
)
cbar.set_label("Combined quality window", labelpad=6)
cbar.ax.xaxis.set_ticks_position("top")
cbar.ax.xaxis.set_label_position("top")
cbar.ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)

fig.supxlabel("Rgt")
fig.supylabel("Ca")
fig.show()

### Profiles inside and outside quality window

In [ ]:
from scipy.spatial import cKDTree

# Map each measured input to the nearest point in the combined prediction window.
feature_columns = ["gap_to_thickness_ratio", "capillary_number"]
profile_window = np.zeros(len(X_unique), dtype=bool)

for slurry in unique_slurries:
    is_prediction = (slurries == slurry).to_numpy()
    is_profile = X_unique.index.get_level_values("slurry") == slurry

    prediction_points = Xpred.loc[is_prediction, feature_columns].to_numpy()
    profile_points = X_unique.loc[is_profile, feature_columns].to_numpy()
    _, nearest_prediction = cKDTree(prediction_points).query(profile_points)
    profile_window[is_profile] = combined_window[is_prediction].to_numpy()[
        nearest_prediction
    ]

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True, constrained_layout=True)
for ax, mask, title, color in zip(
    axes,
    [profile_window, ~profile_window],
    ["Inside combined quality window", "Outside combined quality window"],
    ["tab:blue", "tab:red"],
):
    ax.plot(np.arange(Ys.shape[1]), Ys[mask].T, color=color, alpha=0.35)
    ax.set_title(f"{title} (n={mask.sum()})")
    ax.set_xlabel("Profile position")

axes[0].set_ylabel("Normalized profile Y")
fig.show()

## GPR (Probabilistic)

In [ ]:
marginal_gpr = pd.read_csv(
    "../../benchmarks/v1/gpr.marginal.Xpred_2D.csv",
    index_col=["index", "batch", "sample"],
)
joint_gpr = pd.read_csv(
    "../../benchmarks/v1/gpr.joint_probability.Xpred_2D.csv",
    index_col=["index", "batch", "sample"],
)

In [ ]:
N_COLORS = 8
cmap = mcolors.LinearSegmentedColormap.from_list(
    "gray_blue",
    [mcolors.to_rgba("gray", alpha=0.3), mcolors.to_rgba("tab:blue", alpha=0.8)],
    N=N_COLORS,
)
levels = np.linspace(0, 1, N_COLORS + 1)
norm = mcolors.BoundaryNorm(levels, ncolors=N_COLORS)


def plot_probability_2d(probability, label):
    fig, axes = plt.subplots(1, len(unique_slurries), sharex=True, sharey=True)
    axes = np.atleast_1d(axes)

    for slurry, ax in zip(unique_slurries, axes):
        ok_pred = slurries == slurry
        prediction_index = np.flatnonzero(ok_pred.to_numpy())
        this_Xpred_df = Xpred[ok_pred]
        this_Xpred = this_Xpred_df.to_xarray().to_array().values
        this_prob = probability.reindex(prediction_index).to_numpy()
        delaunay = Delaunay[ok_pred.to_numpy()].values.reshape(this_Xpred.shape[1:])

        ax.contourf(
            this_Xpred[0, ...].squeeze(axis=-1),
            this_Xpred[1, ...].squeeze(axis=-1),
            this_prob.reshape(this_Xpred.shape[1:]).squeeze(axis=-1),
            cmap=cmap,
            norm=norm,
            levels=levels,
        )
        ax.contour(
            this_Xpred[0, ...].squeeze(axis=-1),
            this_Xpred[1, ...].squeeze(axis=-1),
            delaunay.squeeze(axis=-1).astype(float),
            levels=[0.5],
            colors="k",
        )
        (cos_theta,) = this_Xpred_df["cosine_of_contact_angle"].unique()
        ax.set_title(f"Cos θ={cos_theta:.2f}")

    fig.tight_layout(rect=[0.02, 0.05, 1.0, 0.8])
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar_ax = fig.add_axes([0.18, 0.82, 0.64, 0.04])
    cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal")
    cbar.set_label(label, labelpad=6)
    cbar.ax.xaxis.set_ticks_position("top")
    cbar.ax.xaxis.set_label_position("top")
    cbar.ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)
    cbar.ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x:.2f}"))
    fig.supxlabel("Rgt")
    fig.supylabel("Ca")
    fig.show()

### Probability window from the H marginal

In [ ]:
marginal_H_gpr = (
    marginal_gpr.loc[marginal_gpr["target"] == "H", "marginal_prob"]
    .groupby(level="index")
    .mean()
)
plot_probability_2d(marginal_H_gpr, f"GPR probability window: P(H < {H_THRESHOLD:g})")

### Probability window from the phi_1 marginal

In [ ]:
marginal_phi_gpr = (
    marginal_gpr.loc[marginal_gpr["target"] == "phi_1", "marginal_prob"]
    .groupby(level="index")
    .mean()
)
plot_probability_2d(
    marginal_phi_gpr, f"GPR probability window: P(φ₁ < {phi_THRESHOLD:g})"
)

### Probability window from the joint probability

In [ ]:
joint_probability_gpr = joint_gpr["joint_prob"].groupby(level="index").mean()
plot_probability_2d(
    joint_probability_gpr,
    f"GPR probability window: P(H < {H_THRESHOLD:g}, φ₁ < {phi_THRESHOLD:g})",
)

## GPQR

In [ ]:
marginal = pd.read_csv(
    "../../benchmarks/v1/gpqr.marginal.Xpred_2D.csv",
    index_col=["index", "batch", "sample"],
)

In [ ]:
N_COLORS = 8
cmap = mcolors.LinearSegmentedColormap.from_list(
    "gray_blue",
    [mcolors.to_rgba("gray", alpha=0.3), mcolors.to_rgba("tab:blue", alpha=0.8)],
    N=N_COLORS,
)

levels = np.linspace(0, 1, N_COLORS + 1)
norm = mcolors.BoundaryNorm(levels, ncolors=N_COLORS)

### Marginal probability (H)

In [ ]:
marginal_H = marginal[marginal["target"] == "H"].drop(columns=["target"])
marginal_H = marginal_H.groupby(level=["index"]).mean()

In [ ]:
fig, axes = plt.subplots(1, len(slurry_map), sharex=True, sharey=True)

for slurry, ax in zip(unique_slurries, axes):
    ok_pred = slurries == slurry

    this_Xpred = Xpred[ok_pred].to_xarray().to_array().values
    this_prob = marginal_H[ok_pred.values].values

    delaunay = Delaunay[ok_pred.values].values.reshape(this_Xpred.shape[1:])
    delaunay_masked = np.full_like(delaunay, np.nan, dtype=float)

    ax.contourf(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        this_prob.reshape(this_Xpred.shape[1:]).squeeze(axis=-1),
        cmap=cmap,
        norm=norm,
        levels=levels,
    )

    ax.contour(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        delaunay.squeeze(axis=-1).astype(float),
        levels=[0.5],
        colors="k",
    )

    ax.set_title(slurry)

fig.tight_layout(rect=[0.02, 0.05, 1.0, 0.8])

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar_ax = fig.add_axes([0.18, 0.82, 0.64, 0.04])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal")
cbar.set_label("Marginal probability (H)", labelpad=6)
cbar.ax.xaxis.set_ticks_position("top")
cbar.ax.xaxis.set_label_position("top")
cbar.ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)
cbar.ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x:.2f}"))

fig.supxlabel("Rgt")
fig.supylabel("Ca")

fig.show()

### Marginal probability (phi_1)

In [ ]:
marginal_phi = marginal[marginal["target"] == "phi_1"].drop(columns=["target"])
marginal_phi = marginal_phi.groupby(level=["index"]).mean()

In [ ]:
fig, axes = plt.subplots(1, len(slurry_map), sharex=True, sharey=True)

for slurry, ax in zip(unique_slurries, axes):
    ok_pred = slurries == slurry

    this_Xpred = Xpred[ok_pred].to_xarray().to_array().values
    this_prob = marginal_phi[ok_pred.values].values

    delaunay = Delaunay[ok_pred.values].values.reshape(this_Xpred.shape[1:])
    delaunay_masked = np.full_like(delaunay, np.nan, dtype=float)

    ax.contourf(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        this_prob.reshape(this_Xpred.shape[1:]).squeeze(axis=-1),
        cmap=cmap,
        norm=norm,
        levels=levels,
    )

    ax.contour(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        delaunay.squeeze(axis=-1).astype(float),
        levels=[0.5],
        colors="k",
    )

    ax.set_title(slurry)

fig.tight_layout(rect=[0.02, 0.05, 1.0, 0.8])

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar_ax = fig.add_axes([0.18, 0.82, 0.64, 0.04])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal")
cbar.set_label("Marginal probability (phi_1)", labelpad=6)
cbar.ax.xaxis.set_ticks_position("top")
cbar.ax.xaxis.set_label_position("top")
cbar.ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)
cbar.ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x:.2f}"))

fig.supxlabel("Rgt")
fig.supylabel("Ca")

fig.show()

### Joint probability

In [ ]:
joint = pd.read_csv(
    "../../benchmarks/v1/gpqr.joint_probability.Xpred_2D.csv",
    index_col=["index", "batch", "sample"],
)
joint = joint.groupby(level=["index"]).mean()

In [ ]:
fig, axes = plt.subplots(1, len(slurry_map), sharex=True, sharey=True)

for slurry, ax in zip(unique_slurries, axes):
    ok_pred = slurries == slurry

    this_Xpred = Xpred[ok_pred].to_xarray().to_array().values
    this_prob = joint[ok_pred.values].values

    delaunay = Delaunay[ok_pred.values].values.reshape(this_Xpred.shape[1:])
    delaunay_masked = np.full_like(delaunay, np.nan, dtype=float)

    ax.contourf(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        this_prob.reshape(this_Xpred.shape[1:]).squeeze(axis=-1),
        cmap=cmap,
        norm=norm,
        levels=levels,
    )

    ax.contour(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        delaunay.squeeze(axis=-1).astype(float),
        levels=[0.5],
        colors="k",
    )

    ax.set_title(slurry)

fig.tight_layout(rect=[0.02, 0.05, 1.0, 0.8])

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar_ax = fig.add_axes([0.18, 0.82, 0.64, 0.04])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal")
cbar.set_label("Joint probability", labelpad=6)
cbar.ax.xaxis.set_ticks_position("top")
cbar.ax.xaxis.set_label_position("top")
cbar.ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)
cbar.ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x:.2f}"))

fig.supxlabel("Rgt")
fig.supylabel("Ca")

fig.show()

### Uncertainty of joint probability

In [ ]:
Xpred = pd.read_csv("../../_temp/v1/Xpred_1D.csv", index_col=[0, 1, 2])

joint = pd.read_csv(
    "../../benchmarks/v1/gpqr.joint_probability.Xpred_1D.csv",
    index_col=["index", "sample"],
).drop(columns=["batch"])
joint_mean = joint.groupby(level=["index"]).mean()
joint_interval = joint.groupby(level=["index"]).quantile([0.025, 0.975])

In [ ]:
columns = ["slurry", "cosine_of_contact_angle"]
cos_thetas = X["cosine_of_contact_angle"].reset_index()[columns]
slurry_map = cos_thetas.drop_duplicates().set_index("cosine_of_contact_angle")["slurry"]

slurries = Xpred["cosine_of_contact_angle"].map(slurry_map)
unique_slurries = [s for s in SLURRIES if s in slurries.unique()]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.ticker import ScalarFormatter

Rgt_pred = Xpred["gap_to_thickness_ratio"].unique()
Cas = Xpred["capillary_number"].unique()

Cas_sorted = np.sort(Cas)
cmap = plt.get_cmap("viridis", len(Cas_sorted))
norm = mcolors.BoundaryNorm(
    np.concatenate(
        [
            [Cas_sorted[0] * 0.9],
            (Cas_sorted[:-1] + Cas_sorted[1:]) / 2,
            [Cas_sorted[-1] * 1.1],
        ]
    ),
    ncolors=len(Cas_sorted),
)

In [ ]:
fig, axes = plt.subplots(1, len(slurry_map), sharex=True, sharey="row")

for slurry, ax in zip(unique_slurries, axes):
    ok = X.index.get_level_values("slurry") == slurry
    this_X = X[ok]

    ok_pred = slurries == slurry
    this_Xpred = Xpred[ok_pred].copy()
    this_Xpred["prediction_index"] = np.flatnonzero(ok_pred)

    for ca in this_X["capillary_number"].unique():
        ok = this_X["capillary_number"] == ca
        ok_pred = this_Xpred["capillary_number"] == ca

        prediction_index = this_Xpred.loc[ok_pred, "prediction_index"]

        prob_mean = joint_mean.loc[prediction_index]
        prob_mean = prob_mean["joint_prob"]
        ax.plot(
            this_Xpred.loc[ok_pred, "gap_to_thickness_ratio"],
            prob_mean,
            color=cmap(norm(ca)),
            label=f"Ca={ca:.3f}",
        )

        prob_interval = joint_interval.loc[prediction_index]
        prob_interval = prob_interval["joint_prob"].unstack(level=-1)
        ax.fill_between(
            this_Xpred.loc[ok_pred, "gap_to_thickness_ratio"],
            prob_interval[0.025],
            prob_interval[0.975],
            facecolor=cmap(norm(ca)),
            edgecolor="none",
            alpha=0.3,
        )

    (cos_theta,) = this_X["cosine_of_contact_angle"].unique()
    ax.set_title(f"Cos θ={cos_theta:.2f}")

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(
    sm, ax=axes, orientation="horizontal", location="top", pad=0.2, aspect=30
)
cbar.set_label("Ca")
quartile_vals = np.quantile(Cas_sorted, [0, 0.25, 0.5, 0.75, 1.0])
nearest_cas = [Cas_sorted[np.argmin(np.abs(Cas_sorted - q))] for q in quartile_vals]
cbar.set_ticks([round(ca, 3) for ca in nearest_cas])
cbar.ax.xaxis.set_major_formatter(ScalarFormatter())

fig.supxlabel("Rgt")
fig.supylabel("Probability")

### Level-set acquisition

Let $L(x)$ and $U(x)$ be the 5th and 95th percentiles of the GPQR joint-probability posterior. For the target level $\tau=0.9$, use the level-set ambiguity acquisition score

$$a(x)=\min\{U(x)-\tau,\;\tau-L(x)\}.$$

The score is positive when the 90% credible interval straddles $\tau$, and it is largest at uncertain points close to the target level. The acquisition is evaluated over the full prediction grid; the Delaunay boundary is shown only as a reference. If no finite score is available, the candidate table and the associated posterior plot are empty.

In [ ]:
TAU = 0.9
CREDIBLE_INTERVAL = (0.05, 0.95)

Xpred_acquisition = pd.read_csv("../../_temp/v1/Xpred_2D.csv", index_col=[0, 1, 2])
Delaunay_acquisition = pd.read_csv(
    "../../_temp/v1/delaunay.Xpred_2D.csv", index_col=[0, 1, 2]
)
joint_acquisition = pd.read_csv(
    "../../benchmarks/v1/gpqr.joint_probability.Xpred_2D.csv",
    index_col=["index", "batch", "sample"],
)

joint_samples = (
    joint_acquisition["joint_prob"]
    .groupby(level=["index", "sample"])
    .mean()
    .unstack("sample")
)
joint_samples_for_quantile = joint_samples.assign(_empty_guard=np.nan)
lower_probability = joint_samples_for_quantile.quantile(CREDIBLE_INTERVAL[0], axis=1)
upper_probability = joint_samples_for_quantile.quantile(CREDIBLE_INTERVAL[1], axis=1)
level_set_score = np.minimum(upper_probability - TAU, TAU - lower_probability)

candidate_score = level_set_score.dropna().nlargest(1).rename("acquisition_score")
candidate_indices = candidate_score.index.to_numpy(dtype=int)
next_candidate = Xpred_acquisition.iloc[candidate_indices].copy()
next_candidate.insert(0, "prediction_index", candidate_indices)
next_candidate["lower_probability"] = lower_probability.reindex(
    candidate_indices
).to_numpy()
next_candidate["upper_probability"] = upper_probability.reindex(
    candidate_indices
).to_numpy()
next_candidate["acquisition_score"] = candidate_score.to_numpy()
next_candidate

In [ ]:
slurries_acquisition = Xpred_acquisition["cosine_of_contact_angle"].map(slurry_map)
unique_slurries_acquisition = [
    slurry for slurry in SLURRIES if slurry in slurries_acquisition.unique()
]

score_range = level_set_score.agg(["min", "max"]).fillna(
    pd.Series({"min": -np.finfo(float).eps, "max": np.finfo(float).eps})
)
score_min = min(score_range["min"], -np.finfo(float).eps)
score_max = max(score_range["max"], np.finfo(float).eps)
score_levels = np.concatenate(
    [
        np.linspace(score_min, 0, 9),
        np.linspace(0, score_max, 9)[1:],
    ]
)
score_norm = mcolors.TwoSlopeNorm(vmin=score_min, vcenter=0, vmax=score_max)
score_cmap = plt.get_cmap("coolwarm")

fig, axes = plt.subplots(1, len(unique_slurries_acquisition), sharex=True, sharey=True)
axes = np.atleast_1d(axes)

for slurry, ax in zip(unique_slurries_acquisition, axes):
    ok_pred = slurries_acquisition == slurry
    prediction_indices = np.flatnonzero(ok_pred.to_numpy())
    this_Xpred_df = Xpred_acquisition[ok_pred]
    this_Xpred = this_Xpred_df.to_xarray().to_array().values
    this_shape = this_Xpred.shape[1:]
    this_score = level_set_score.reindex(prediction_indices).to_numpy()
    this_delaunay = (
        Delaunay_acquisition[ok_pred.to_numpy()].to_numpy().reshape(this_shape)
    )
    contour = ax.contourf(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        this_score.reshape(this_shape).squeeze(axis=-1),
        levels=score_levels,
        cmap=score_cmap,
        norm=score_norm,
        extend="both",
    )
    ax.contour(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        this_delaunay.squeeze(axis=-1).astype(float),
        levels=[0.5],
        colors="k",
    )

    (cos_theta,) = this_Xpred_df["cosine_of_contact_angle"].unique()
    this_candidate = next_candidate[
        next_candidate["cosine_of_contact_angle"] == cos_theta
    ]
    ax.scatter(
        this_candidate["gap_to_thickness_ratio"],
        this_candidate["capillary_number"],
        marker="*",
        s=120,
        facecolor="gold",
        edgecolor="k",
        linewidth=0.8,
        zorder=3,
    )
    ax.set_title(f"Cos θ={cos_theta:.2f}")

cbar = fig.colorbar(contour, ax=axes, orientation="horizontal", location="top", pad=0.2)
cbar.set_label(f"Level-set ambiguity score (τ={TAU:g}; positive = unresolved)")
fig.supxlabel("Rgt")
fig.supylabel("Ca")
fig.show()

#### Joint-probability posterior at the selected point

In [ ]:
selected_probability_samples = joint_samples.reindex(candidate_indices).stack().dropna()
selected_probability_mean = selected_probability_samples.mean()
selected_probability_lower = selected_probability_samples.quantile(CREDIBLE_INTERVAL[0])
selected_probability_upper = selected_probability_samples.quantile(CREDIBLE_INTERVAL[1])

n_bins = min(20, max(5, int(np.ceil(np.sqrt(len(selected_probability_samples))))))
fig, ax = plt.subplots()
ax.hist(
    selected_probability_samples,
    bins=np.linspace(0, 1, n_bins + 1),
    density=False,
    color="tab:blue",
    alpha=0.55,
    edgecolor="white",
)
ax.plot(
    selected_probability_samples,
    np.full(len(selected_probability_samples), 0.02),
    "|",
    color="tab:blue",
    markersize=10,
    transform=ax.get_xaxis_transform(),
)
ax.axvline(TAU, color="tab:red", linewidth=2, label=f"τ={TAU:g}")
ax.axvline(
    selected_probability_mean,
    color="k",
    linestyle="--",
    label="Posterior mean",
)
ax.axvline(
    selected_probability_lower,
    color="tab:orange",
    linestyle=":",
    label="90% credible interval",
)
ax.axvline(selected_probability_upper, color="tab:orange", linestyle=":")
ax.set(
    xlim=(0, 1),
    xlabel="Joint probability",
    ylabel="Posterior sample count",
    title="Joint-probability posterior at the selected point",
)
ax.legend()
fig.tight_layout()
fig.show()